# Catchment and stream network delineation for each dam

Delineates one catchment and its stream network per dam outlet from the conditioned NASADEM DEM,
using pysheds D8 flow routing. All parameters are read from `config/dams.yaml`.
Results are written to `outputs/drainage.gpkg`.

## Load config and DEM

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import shapes
from rasterio.transform import array_bounds
import yaml
from pyproj import CRS, Transformer
from pysheds.grid import Grid
from pysheds.sview import Raster
from shapely.geometry import Point, shape
from shapely.ops import unary_union

ROOT = Path.cwd().parent
CONFIG_PATH = ROOT / "config" / "dams.yaml"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

dem_path = ROOT / "data" / f"dem_{config['dem']['source'].lower()}_utm.tif"

In [ ]:
# Read the nodata value stored in the DEM file
with rasterio.open(dem_path) as src:
    dem_nodata = src.nodata
    dem_crs = CRS.from_user_input(src.crs)

# Load the DEM with pysheds, passing the nodata value so those cells are excluded from routing
grid = Grid.from_raster(str(dem_path), nodata=dem_nodata)
dem = grid.read_raster(str(dem_path), nodata=dem_nodata)

project_crs = CRS.from_user_input(config["crs"])
print(f"DEM: {dem_path.name}, shape {dem.shape}, nodata {dem_nodata}")
print(f"DEM CRS matches project CRS: {dem_crs == project_crs}")

## Condition the DEM

In [ ]:
# Fill single-cell pits, then larger depressions, then resolve flat areas so every cell drains
pit_filled_dem = grid.fill_pits(dem)
flooded_dem = grid.fill_depressions(pit_filled_dem)
inflated_dem = grid.resolve_flats(flooded_dem)

## Flow direction (D8) and flow accumulation

In [ ]:
fdir = grid.flowdir(inflated_dem, routing="d8")
acc = grid.accumulation(fdir, routing="d8")

## Stream threshold

In [ ]:
# Convert the stream threshold from km² to a number of cells using the DEM pixel area
pixel_area_m2 = abs(grid.affine.a * grid.affine.e)
threshold_km2 = config["stream_threshold_km2"]
threshold_cells = int(round(threshold_km2 * 1e6 / pixel_area_m2))
print(f"Pixel area: {pixel_area_m2:.1f} m²")
print(f"Stream threshold: {threshold_km2} km² = {threshold_cells} cells")

# Grid settings for boolean masks, with nodata set to False instead of the DEM nodata value
mask_view = grid.viewfinder.copy()
mask_view.nodata = False

stream_mask = Raster(np.asarray(acc >= threshold_cells), viewfinder=mask_view)

## Outlets

In [ ]:
names = [d["name"] for d in config["dams"]]
xy = np.array([[d["easting"], d["northing"]] for d in config["dams"]], dtype=float)

# Reproject the dam coordinates to the project CRS if they are given in a different CRS
coords_crs = CRS.from_user_input(config["coords_crs"])
if coords_crs != project_crs:
    transformer = Transformer.from_crs(coords_crs, project_crs, always_xy=True)
    xy[:, 0], xy[:, 1] = transformer.transform(xy[:, 0], xy[:, 1])

# Snap each outlet to the nearest cell at or above the stream threshold.
# snap_to_mask returns the top-left corner of that cell, so shift it to the cell centre.
corner_xy = grid.snap_to_mask(stream_mask, xy)
snapped_xy = corner_xy + np.array([grid.affine.a / 2, grid.affine.e / 2])
snap_dist_m = np.hypot(*(snapped_xy - xy).T)
flagged = snap_dist_m > config["max_snap_distance_m"]

# Row and column of each snapped outlet cell
outlet_cells = [grid.nearest_cell(x, y, snap="center") for x, y in snapped_xy]

for name, dist, flag in zip(names, snap_dist_m, flagged):
    print(f"{name}: snapped {dist:.1f} m" + ("  FLAGGED" if flag else ""))

## Catchments

In [ ]:
# Cells that are nodata in the DEM
nodata_cells = np.asarray(dem == dem_nodata)


def touches_edge_or_nodata(mask):
    """Return True if the mask reaches the DEM border or is next to a nodata cell."""
    if mask[0, :].any() or mask[-1, :].any() or mask[:, 0].any() or mask[:, -1].any():
        return True
    # Grow the mask by one cell in all 8 directions and test for overlap with nodata
    padded = np.pad(mask, 1)
    grown = np.zeros_like(mask)
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            grown |= padded[1 + dr : 1 + dr + mask.shape[0], 1 + dc : 1 + dc + mask.shape[1]]
    return bool((grown & nodata_cells).any())


catchment_masks = {}
catchment_records = []
for name, (x, y) in zip(names, snapped_xy):
    catch = grid.catchment(x=x, y=y, fdir=fdir, routing="d8", xytype="coordinate", snap="center")
    mask = np.asarray(catch).astype(bool)
    catchment_masks[name] = mask

    # Outlets of the other dams that fall inside this catchment
    contained = [
        other for other, (col, row) in zip(names, outlet_cells)
        if other != name and mask[row, col]
    ]

    # Merge the polygons of all catchment cells into one geometry
    polygons = [
        shape(geom)
        for geom, value in shapes(mask.astype(np.uint8), mask=mask, transform=grid.affine)
        if value == 1
    ]
    catchment_records.append({
        "name": name,
        "area_km2": mask.sum() * pixel_area_m2 / 1e6,
        "touches_edge": touches_edge_or_nodata(mask),
        "contains_outlets": ", ".join(contained),
        "geometry": unary_union(polygons),
    })

catchments = gpd.GeoDataFrame(catchment_records, geometry="geometry", crs=project_crs)
catchments.drop(columns="geometry")

## Streams

In [ ]:
# Extract the stream network inside each catchment from cells at or above the stream threshold
stream_frames = []
for name, mask in catchment_masks.items():
    catchment_streams = Raster(np.asarray(stream_mask) & mask, viewfinder=mask_view)
    network = grid.extract_river_network(fdir, catchment_streams, routing="d8")
    segments = gpd.GeoDataFrame.from_features(network["features"], crs=project_crs)
    segments = segments[["geometry"]].assign(name=name)
    stream_frames.append(segments)

streams = gpd.GeoDataFrame(
    pd.concat(stream_frames, ignore_index=True), geometry="geometry", crs=project_crs
)[["name", "geometry"]]
streams.groupby("name").size().rename("segments")

## Save to GeoPackage

In [ ]:
outlets = gpd.GeoDataFrame(
    {
        "name": names,
        "orig_easting": xy[:, 0],
        "orig_northing": xy[:, 1],
        "snap_easting": snapped_xy[:, 0],
        "snap_northing": snapped_xy[:, 1],
        "snap_dist_m": snap_dist_m,
        "flagged": flagged,
    },
    geometry=[Point(x, y) for x, y in snapped_xy],
    crs=project_crs,
)

gpkg_path = OUTPUT_DIR / "drainage.gpkg"
gpkg_path.unlink(missing_ok=True)
outlets.to_file(gpkg_path, layer="outlets", driver="GPKG")
catchments.to_file(gpkg_path, layer="catchments", driver="GPKG")
streams.to_file(gpkg_path, layer="streams", driver="GPKG")
print(f"Saved {gpkg_path}")
outlets.drop(columns="geometry")

## Quick-look plot

In [ ]:
# Show the DEM, catchment outlines, streams and snapped outlets, zoomed to the catchments
left, bottom, right, top = array_bounds(*dem.shape, grid.affine)
dem_display = np.where(nodata_cells, np.nan, np.asarray(dem))

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(dem_display, extent=(left, right, bottom, top), cmap="terrain")
catchments.boundary.plot(ax=ax, color="black", linewidth=1)
streams.plot(ax=ax, color="tab:blue", linewidth=0.6)
outlets.plot(ax=ax, color="red", markersize=30, zorder=3)
for name, (x, y) in zip(names, snapped_xy):
    ax.annotate(name, (x, y), xytext=(5, 5), textcoords="offset points")

xmin, ymin, xmax, ymax = catchments.total_bounds
margin = 0.05 * max(xmax - xmin, ymax - ymin)
ax.set_xlim(xmin - margin, xmax + margin)
ax.set_ylim(ymin - margin, ymax + margin)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title("Dam catchments and stream networks")
plt.show()